In [1]:
!git clone https://github.com/cs329yangzhong/discourse-driven-summary-factuality-evaluation
%cd discourse-driven-summary-factuality-evaluation
!pip install -q fuzzywuzzy python-Levenshtein nltk networkx

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

Cloning into 'discourse-driven-summary-factuality-evaluation'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 90 (delta 19), reused 82 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 6.08 MiB | 6.39 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/discourse-driven-summary-factuality-evaluation


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
!python structscore/run_diversumm.py --systems segment_minicheck_score segment_minicheck_lvl_1 align_score

segment_minicheck_score
origins = ['multinews', 'qmsum', 'GovReport', 'arXiv', 'ChemSum']
/content/discourse-driven-summary-factuality-evaluation/structscore/discourse_utils.py:586: RuntimeWarning: invalid value encountered in sqrt
  depth = int(np.sqrt(int(end_id) - int(start_id)))
old: [np.float64(61.9), np.float64(60.66), np.float64(83.24), np.float64(78.66), np.float64(59.74)] 68.84
reweighted: [np.float64(57.85), np.float64(58.12), np.float64(86.74), np.float64(79.52), np.float64(60.71)] 68.588

segment_minicheck_lvl_1
origins = ['multinews', 'qmsum', 'GovReport', 'arXiv', 'ChemSum']
/content/discourse-driven-summary-factuality-evaluation/structscore/discourse_utils.py:586: RuntimeWarning: invalid value encountered in sqrt
  depth = int(np.sqrt(int(end_id) - int(start_id)))
old: [np.float64(45.33), np.float64(61.1), np.float64(87.5), np.float64(85.22), np.float64(63.59)] 68.548
reweighted: [np.float64(39.15), np.float64(57.82), np.float64(88.46), np.float64(87.7), np.float64(62.55

In [4]:
import sys, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
sys.path.append('structscore')
import discourse_utils as du

df = pd.read_csv('data/diversumm_test.csv')
splits = ['multinews', 'qmsum', 'GovReport', 'arXiv', 'ChemSum']
names  = ['MNW', 'QMS', 'GOV', 'AXV', 'CSM']

# Paper Table 4 values: (column, paper row, original AUCs, re-weighted AUCs)
targets = [
    ('segment_minicheck_score', 'Row 11/12 MC-FT5 (SENT)',
        [57.67, 60.66, 83.24, 78.66, 59.74], [56.79, 60.36, 84.75, 79.38, 60.06]),
    ('segment_minicheck_lvl_1', 'Row 13/14 + LV1 / StructS-LV1',
        [44.80, 61.10, 87.50, 85.22, 63.59], [38.45, 60.66, 88.05, 86.32, 63.11]),
]

rows = []
for col, label, paper_orig, paper_rw in targets:
    for i, split in enumerate(splits):
        d = df[df.origin == split]
        orig, rw, fails = [], [], 0
        for _, r in d.iterrows():
            scores = du.process_sent_score(str(r[col]))
            try:
                w, o, *_ = du.reweight_score(scores, r, alpha=1, depth_factor=1)
            except Exception:
                o = w = float(np.mean(scores)); fails += 1
            orig.append(o); rw.append(w)
        rows.append({
            'system': label, 'split': names[i],
            'paper_orig': paper_orig[i], 'mine_orig': round(roc_auc_score(d.label, orig)*100, 2),
            'paper_rw':   paper_rw[i],   'mine_rw':   round(roc_auc_score(d.label, rw)*100, 2),
            'align_failures': fails,
        })

res = pd.DataFrame(rows)
res['diff_rw'] = (res.mine_rw - res.paper_rw).round(2)
pd.set_option('display.width', 200)
print(res.to_string(index=False))
print('\nMacro averages:')
print(res.groupby('system')[['paper_orig','mine_orig','paper_rw','mine_rw']].mean().round(2))

/content/discourse-driven-summary-factuality-evaluation/structscore/discourse_utils.py:586: RuntimeWarning: invalid value encountered in sqrt
  depth = int(np.sqrt(int(end_id) - int(start_id)))
/content/discourse-driven-summary-factuality-evaluation/structscore/discourse_utils.py:586: RuntimeWarning: invalid value encountered in sqrt
  depth = int(np.sqrt(int(end_id) - int(start_id)))


                       system split  paper_orig  mine_orig  paper_rw  mine_rw  align_failures  diff_rw
      Row 11/12 MC-FT5 (SENT)   MNW       57.67      61.90     56.79    57.85               0     1.06
      Row 11/12 MC-FT5 (SENT)   QMS       60.66      60.66     60.36    58.12               0    -2.24
      Row 11/12 MC-FT5 (SENT)   GOV       83.24      83.24     84.75    86.74               0     1.99
      Row 11/12 MC-FT5 (SENT)   AXV       78.66      78.66     79.38    79.52               0     0.14
      Row 11/12 MC-FT5 (SENT)   CSM       59.74      59.74     60.06    60.71               0     0.65
Row 13/14 + LV1 / StructS-LV1   MNW       44.80      45.33     38.45    39.15               0     0.70
Row 13/14 + LV1 / StructS-LV1   QMS       61.10      61.10     60.66    57.82               0    -2.84
Row 13/14 + LV1 / StructS-LV1   GOV       87.50      87.50     88.05    88.46               0     0.41
Row 13/14 + LV1 / StructS-LV1   AXV       85.22      85.22     86.32    8

In [5]:
res.to_csv('reproduction_results.csv', index=False)
from google.colab import files
files.download('reproduction_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>